In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 4, Finished, Available, Finished, False)

### Reading Shortcut from the Silver Lakehouse

In [3]:
df = spark.read.format("csv").option("header", True).load("Files/raw_dataAW_Customers/AW_Customers.csv")
display(df.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 879f9d33-954b-4f0f-818b-a8f6adc8f997)

### Reading from the Bronze Lakehouse (Fabric_LH)

#### 1 - Reading 'customers' data

In [4]:
df1 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_dataAW_Customers")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 6, Finished, Available, Finished, False)

#### Using 'inferschema' to make the csv file retain the correct data-type of the respective columns 
###### (otherwise only 'string' data-type can be seen in every column)

In [5]:
df1.printSchema()

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 7, Finished, Available, Finished, False)

root
 |-- CustomerKey: integer (nullable = true)
 |-- Prefix: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- BirthDate: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- AnnualIncome: string (nullable = true)
 |-- TotalChildren: integer (nullable = true)
 |-- EducationLevel: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- HomeOwner: string (nullable = true)



In [6]:
display(df1.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd7ca3f2-1fdc-44f7-9835-f11b54686d8e)

#### 2 - Data Transformation 

##### (a) -- creating new column for 'Full Name'

In [7]:
df1 = df1.withColumn('FullName', concat(col('Prefix'), lit(" "), col('FirstName'), lit(" "), col('LastName')))
display(df1.limit(20))

# lit(" ") is used when we insert a constant, and here the constant is 'space'.

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87033b61-0e50-453a-bb67-a197653efcf7)

#### (b) -- splitting the 'EmailAddress' column

In [8]:
df1 = df1.withColumn("Domain", split("EmailAddress", "@",)[1])
display(df1.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4746b4f-c2c8-488b-8700-273283d35133)

In [9]:
df1.write.format("delta")\
    .mode("append")\
    .option("path", "Files/customers")\
    .saveAsTable("customers")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 11, Finished, Available, Finished, False)

#### 3 - Reading 'sales' data

In [10]:
df2 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_dataAdventureWorks_Sales_2016")


df3 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_dataAdventureWorks_Sales_2017")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 12, Finished, Available, Finished, False)

In [11]:
df4 = df2.union(df3)
display(df4.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 17b5e829-75da-4145-bda9-69a255a30d3f)

In [12]:
df4.write.format("delta")\
    .mode("append")\
    .option("path", "Files/sales")\
    .saveAsTable("sales")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 14, Finished, Available, Finished, False)

#### 4 - Reading 'product sub-category' data from the merged table created before

In [13]:
df5 = spark.read.format("delta")\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Tables/dbo/Merge_PC_PSC")

display(df5.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70d2840b-4b73-4f7c-b26b-a8f913bd88bb)

In [14]:
df5.write.format("delta")\
    .mode("append")\
    .option("path", "Files/prod_subcat")\
    .saveAsTable("prod_subcat")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 16, Finished, Available, Finished, False)

#### 5 - Reading 'calendar' data

In [16]:
df6 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_dataAdventureWorks_Calendar")


StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 18, Finished, Available, Finished, False)

In [17]:
display(df6.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, babb8203-98ad-4175-8d93-8857a740c0f8)

In [18]:
df6.write.format("delta")\
    .mode("append")\
    .option("path", "Files/calendar")\
    .saveAsTable("calendar")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 20, Finished, Available, Finished, False)

#### 6 - Reading 'product' data

In [19]:
df7 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_dataAdventureWorks_Products")

display(df7.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 493311c5-ea84-4429-85a6-d7ef00b3715d)

In [21]:
# Splitting the 'ProductSKU' columns to get the category

df7 = df7.withColumn("ProductSKU", split("ProductSKU", "-",)[0])
display(df7.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2fe777b-e8d8-4c63-a94e-a78871ac0c23)

In [22]:
df7.write.format("delta")\
    .mode("append")\
    .option("path", "Files/products")\
    .saveAsTable("products")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 24, Finished, Available, Finished, False)

#### 7 - Reading 'returns' data from ADLS

In [23]:
df8 = spark.read.format("csv")\
    .option("header", True)\
    .option("inferschema", True)\
    .load("abfss://Snklp_Fabric_WS@onelake.dfs.fabric.microsoft.com/Fabric_LH.Lakehouse/Files/raw_data_ADLS")

display(df8.limit(20))

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7246b6ea-3768-4ef1-9d52-982596bdae58)

##### - The created table goes into **delta-parquet** format by default if we create them from **'Managed Table'**.
##### - But since the 'returns' table is from **'external storage'**, we can decide into which format the table should be written.

In [24]:
df7.write.format("csv")\
    .mode("append")\
    .option("path", "Files/returns")\
    .saveAsTable("returns")

StatementMeta(, f748f3cb-3673-4206-8aeb-1d4f7500d8f7, 26, Finished, Available, Finished, False)